# V8 · Stage 1.1b — Rebuild dataset with trajectory-grouped splits

**Roadmap reference**: `paper/research_roadmap.md` · Stage 1 · Task 1.1b
**Strategy plan**: `pybamm_research_strengthening_plan.md`
**Blocking predecessor**: `01_1_grouped_split_audit.ipynb` (verdict: FAIL)

## Acceptance criterion
Zero cross-split leakage on `sim_id`, `base_window`, and SoH-offset copies.
Assertions must pass:

```python
assert df.groupby(["anchor_id", "sample_id"])["split"].nunique().max() == 1
assert df.groupby(["anchor_id", "sample_id", "context_start"])["split"].nunique().max() == 1
```

Temporal-overlap re-audit must return **0** leaked ordered pairs.

## Expected outputs
- `configs/phase3_corpus/_v8_dataset.parquet` — same rows as v7, new split column
- `outputs/results/grouped_split_audit_clean.json` — clean split audit metrics

## Approach
The v7 dataset (`_v7_dataset.parquet`) is preserved unchanged for reproducibility.
The clean split is applied at the `sim_id = (anchor_id, sample_id)` level:

1. Enumerate distinct sims (486 sims across 7 anchors).
2. Stratify by anchor: for each anchor, its sims are split 70/15/15 among train/val/test using `SEED=456`.
3. Every row inherits its sim's split assignment.
4. Assert no leakage on any dimension.
5. Save as `_v8_dataset.parquet`; do **not** overwrite the v7 file.


## 1. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJ = Path("/home/hj/Desktop/PINNs")
V7_DATASET = PROJ / "configs" / "phase3_corpus" / "_v7_dataset.parquet"
V8_DATASET = PROJ / "configs" / "phase3_corpus" / "_v8_dataset.parquet"
OUT_JSON   = PROJ / "outputs" / "results" / "grouped_split_audit_clean.json"
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)

SEED = 456
print(f"seed = {SEED}")
print(f"v7 dataset  = {V7_DATASET}")
print(f"v8 dataset  = {V8_DATASET}")


seed = 456
v7 dataset  = /home/hj/Desktop/PINNs/configs/phase3_corpus/_v7_dataset.parquet
v8 dataset  = /home/hj/Desktop/PINNs/configs/phase3_corpus/_v8_dataset.parquet


## 2. Load rows and enumerate sims

In [2]:
df = pd.read_parquet(V7_DATASET).reset_index(drop=True)
print(f"rows: {len(df):,}")
sims = df[["anchor_id", "sample_id"]].drop_duplicates().reset_index(drop=True)
print(f"distinct sims: {len(sims):,}")
print("sims per anchor:")
print(sims.groupby("anchor_id").size().to_string())


rows: 14,096
distinct sims: 486
sims per anchor:
anchor_id
CALB_0003    70
CALB_0009    70
CALB_0010    70
CALB_0015    70
EVE_0004     70
REPT_0007    68
REPT_0057    68


## 3. Anchor-stratified sim-level split (70/15/15)

Each anchor's sims are shuffled with `SEED=456`, then partitioned 70/15/15 by count.
This preserves anchor representation across all splits (needed for the within-anchor
interpolation experiment), while guaranteeing every simulated trajectory lives in
exactly one split.

In [3]:
rng = np.random.default_rng(SEED)
sim_split = np.empty(len(sims), dtype=object)
for aid in sims["anchor_id"].unique():
    idx = np.array(sims.index[sims["anchor_id"] == aid])
    rng.shuffle(idx)
    n = len(idx)
    n_val  = int(round(n * 0.15))
    n_test = int(round(n * 0.15))
    n_train = n - n_val - n_test
    sim_split[idx[:n_train]] = "train"
    sim_split[idx[n_train:n_train+n_val]] = "val"
    sim_split[idx[n_train+n_val:]] = "test"
sims["split"] = sim_split
print("sim assignments per anchor:")
print(sims.groupby(["anchor_id", "split"]).size().unstack(fill_value=0))


sim assignments per anchor:
split      test  train  val
anchor_id                  
CALB_0003    10     50   10
CALB_0009    10     50   10
CALB_0010    10     50   10
CALB_0015    10     50   10
EVE_0004     10     50   10
REPT_0007    10     48   10
REPT_0057    10     48   10


In [4]:
# Merge split assignment back to every row
df_v8 = df.merge(sims, on=["anchor_id", "sample_id"], how="left")
assert df_v8["split"].notna().all(), "some rows missing split assignment"
print(f"rows per split:")
print(df_v8["split"].value_counts())


rows per split:
split
train    10004
test      2052
val       2040
Name: count, dtype: int64


## 4. Hard assertions — must all pass

In [5]:
assertions = []

# 1. Every (anchor_id, sample_id) belongs to exactly one split
max_splits_per_sim = df_v8.groupby(["anchor_id", "sample_id"])["split"].nunique().max()
assertions.append(("sim_id → single split", max_splits_per_sim, max_splits_per_sim == 1))

# 2. Every (anchor_id, sample_id, context_start) belongs to exactly one split
max_splits_per_base = df_v8.groupby(["anchor_id", "sample_id", "context_start"])["split"].nunique().max()
assertions.append(("base_window → single split", max_splits_per_base, max_splits_per_base == 1))

# 3. SoH-offset copies of a base window belong to exactly one split
#    (implied by (2) but reported explicitly)
max_splits_per_aug = df_v8.groupby(["anchor_id", "sample_id", "context_start", "soh_offset"])["split"].nunique().max()
assertions.append(("(base × soh_offset) → single split", max_splits_per_aug, max_splits_per_aug == 1))

# 4. Anchor stratification: each anchor represented in all splits (matches original v7 stratification)
anchor_split_counts = df_v8.groupby(["anchor_id", "split"]).size().unstack(fill_value=0)
anchor_in_all = int((anchor_split_counts > 0).all(axis=1).sum())
assertions.append((f"each anchor in all 3 splits ({anchor_in_all}/{anchor_split_counts.shape[0]})",
                    anchor_in_all, anchor_in_all == anchor_split_counts.shape[0]))

print(f"{'assertion':<45}  {'value':>8}  {'result':>6}")
for name, val, ok in assertions:
    print(f"{name:<45}  {val!s:>8}  {'PASS' if ok else 'FAIL':>6}")

for _, val, ok in assertions:
    if not ok:
        raise AssertionError(f"assertion FAIL: value={val}")
print("\nall assertions PASS")


assertion                                         value  result
sim_id → single split                                 1    PASS
base_window → single split                            1    PASS
(base × soh_offset) → single split                    1    PASS
each anchor in all 3 splits (7/7)                     7    PASS

all assertions PASS


## 5. Temporal-overlap re-audit — must be 0

Because sims are now sim-disjoint across splits, no two rows in different splits
share a sim_id — so temporal overlap between splits is structurally impossible.
The audit below confirms this.

In [6]:
df_v8["ctx_lo"] = df_v8["context_start"]
df_v8["ctx_hi"] = df_v8["context_start"] + df_v8["K"]
df_v8["tgt_lo"] = df_v8["ctx_hi"]
df_v8["tgt_hi"] = df_v8["tgt_lo"] + df_v8["forecast_len"]

temporal_leaks = 0
for (aid, sid), grp in df_v8.groupby(["anchor_id", "sample_id"]):
    if grp["split"].nunique() < 2:
        continue
    # sim spans multiple splits — check every ordered pair for temporal overlap
    grp = grp.reset_index(drop=True)
    for i in range(len(grp)):
        for j in range(len(grp)):
            if i == j: continue
            if grp.loc[i, "split"] == grp.loc[j, "split"]: continue
            if grp.loc[i, "tgt_lo"] < grp.loc[j, "ctx_hi"] and grp.loc[j, "ctx_lo"] < grp.loc[i, "tgt_hi"]:
                temporal_leaks += 1

print(f"temporal leaked ordered pairs: {temporal_leaks}")
assert temporal_leaks == 0, f"temporal leakage detected: {temporal_leaks} pairs"
print("temporal audit PASS")


temporal leaked ordered pairs: 0
temporal audit PASS


## 6. Persist clean dataset + audit report

In [7]:
# Drop helper columns before persistence
df_v8_out = df_v8.drop(columns=["ctx_lo", "ctx_hi", "tgt_lo", "tgt_hi"])
df_v8_out.to_parquet(V8_DATASET, index=False)
print(f"wrote {V8_DATASET}  ({len(df_v8_out):,} rows)")

report = {
    "input_dataset": str(V7_DATASET),
    "output_dataset": str(V8_DATASET),
    "seed": SEED,
    "n_rows": int(len(df_v8_out)),
    "n_sims": int(len(sims)),
    "split_counts": df_v8_out["split"].value_counts().to_dict(),
    "sims_per_split_per_anchor": (
        sims.groupby(["anchor_id","split"]).size().unstack(fill_value=0).to_dict()
    ),
    "assertions": {
        "sim_id_single_split": True,
        "base_window_single_split": True,
        "aug_copy_single_split": True,
        "temporal_overlap_leaked_pairs": 0,
    },
    "overall_verdict": "PASS — leakage-safe within-anchor trajectory split ready",
}
OUT_JSON.write_text(json.dumps(report, indent=2, default=str))
print(f"wrote {OUT_JSON}")


wrote /home/hj/Desktop/PINNs/configs/phase3_corpus/_v8_dataset.parquet  (14,096 rows)
wrote /home/hj/Desktop/PINNs/outputs/results/grouped_split_audit_clean.json


## 7. Verdict marker

- [x] **PASS** — acceptance criterion fully met.
- [ ] **PASS WITH LIMITATIONS**
- [ ] **FAIL — BLOCKS DOWNSTREAM CLAIM**

**Overall verdict**: `PASS — leakage-safe within-anchor trajectory split ready`

### What this split supports

- validation and test performance on unseen simulated trajectories
- early stopping based on the clean validation split
- baseline comparisons
- no-θ ablation
- multi-seed variance
- within-anchor synthetic generalisation

### What this split does NOT support

- leave-one-anchor-out claims
- leave-one-supplier-out claims
- unseen-supplier generalisation
- strong supplier-independent claims

**Result artifact(s) written**:
- `configs/phase3_corpus/_v8_dataset.parquet`
- `outputs/results/grouped_split_audit_clean.json`

**Notes**:
- Same 14,096 rows as v7; only the `split` column changed.
- 486 sims partitioned 70/15/15 within each anchor.
- Split assertions all pass: sim_id, base_window, and SoH-offset copy dimensions
  all report `max_splits_per_group == 1`.
- Temporal target-context overlap between splits: **0** ordered pairs
  (down from 29,187 in v7).
- Ready for `01_1c_retrain_clean_split.ipynb`.
